## Neural Networks and LLMs

Today's focus bridges the gap between classical feature-engineered models, deep learning architectures, and modern Large Language Models (LLMs) to enhance price estimation performance.

In [3]:
# 1. Standard Library
import csv
import os

# 2. Third-Party Packages
from dotenv import load_dotenv
from huggingface_hub import login
from litellm import completion
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from rich import print

# 3. Local / Project-Specific Modules
from price_agent.data.evaluator import evaluate
from price_agent.data.items import Item

d:\ujjwal\the_capstone_project\.venv\Lib\site-packages\huggingface_hub\constants.py:310: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


In [4]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-08-19 22:17:15,426 WARNING huggingface_hub._login Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
username = "ujjwalsingh108"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items

# Human Baseline Evaluation Workflow

This workflow establishes a **human benchmark** for the price prediction task by exporting test samples for manual labeling, reading back human guesses, and benchmarking them using the custom evaluation suite.

---

### 1. Exporting Unlabeled Test Samples to CSV

```python
# Write the first 100 test items to a CSV file for manual human guessing
with open("human_in.csv", "w", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

```

* **`test[:100]`**: Selects the first 100 items from the `test` split.
* **`writer.writerow([t.summary, 0])`**: Writes each product summary alongside a placeholder price of `0`.
* **Purpose**: Generates a clean spreadsheet (`human_in.csv`) where human evaluators can inspect product summaries and fill in their estimated prices.

---

### 2. Loading Completed Human Predictions

```python
# Read back the CSV containing completed human price guesses
human_predictions = []
with open("human_out.csv", "r", encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

```

* Opens `human_out.csv` (the completed spreadsheet with human guesses in the second column).
* Parses column index `1` as a float and appends it to the `human_predictions` list in sequential order.

---

### 3. Defining the Human Predictor Function

```python
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

```

* **`test.index(item)`**: Locates the index of the queried `Item` within the `test` dataset.
* **`human_predictions[idx]`**: Returns the corresponding human guess for that specific item.

---

### 4. Single-Item Verification

```python
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")

```

* Retrieves and prints a side-by-side comparison of the human guess vs. the ground truth price for the very first test item (`test[0]`).

---

### 5. Benchmark Evaluation

```python
evaluate(human_pricer, test, size=100)

```

* Passes `human_pricer` into `evaluate()`.
* **`size=100`**: Restricts evaluation to the 100 items that were manually scored.
* Computes standard error metrics (MAE, MSE, $R^2$) and renders scatter/trend charts to determine whether machine learning and LLM models can outperform human intuition.

In [6]:
# 2. Write the first 100 test items to human_in.csv
in_path = Path("data/04-predictions/human_labeling/human_in.csv")
in_path.parent.mkdir(parents=True, exist_ok=True)

with in_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    for t in test[:100]:
        writer.writerow([t.summary or "", 0])

print(f"Written {len(test[:100])} rows to {in_path}")

Written 100 rows to data\04-predictions\human_labeling\human_in.csv

In [7]:
# 3. For the purpose of this baseline test, generate a copy as human_out.csv
# (In real testing, a human fills in prices in column 2 of human_out.csv)
out_path = Path("data/04-predictions/human_labeling/human_out.csv")

with out_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    for t in test[:100]:
        # Using a dummy price of 50.0 for initial testing
        writer.writerow([t.summary or "", 50.0])

In [8]:
# 4. Read human_out.csv back in
human_predictions = []
with out_path.open("r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if row:  # skip empty lines
            human_predictions.append(float(row[1]))

print(f"Successfully loaded {len(human_predictions)} human predictions")

Successfully loaded 100 human predictions

In [9]:
# 5. Define predictor and run
def human_pricer(item: Item) -> float:
    idx = test.index(item)
    return human_predictions[idx]

In [10]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted ${human:.2f} for an item that actually costs ${actual:.2f}")

Human predicted $50.00 for an item that actually costs $144.96

In [11]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$95 $24 $34 $24 $10 $138 $3 $15 $39 $225 $349 $282 $31 $14 $690 $35 $18 $5 $31 $10 $29 $150 $120 $20 $0 $31 $60 $21 $30 $44 $45 $35 $21 $34 $30 $242 $15 $14 $83 $35 $30 $82 $32 $38 $20 $37 $37 $34 $14 $1 $36 $20 $184 $4 $24 $20 $43 $38 $34 $6 $23 $5 $17 $30 $95 $15 $34 $179 $14 $37 $4 $38 $31 $23 $30 $37 $5 $42 $36 $23 $10 $27 $42 $7 $17 $36 $40 $840 $17 $22 $34 $32 $31 $20 $14 $29 $34 $34 $37 $41 

# Deep Learning Regression: Vanilla Neural Network with PyTorch

This workflow transitions from classical machine learning to deep learning by training an 8-layer deep **Multi-Layer Perceptron (MLP)** from scratch in PyTorch to predict continuous item prices from text embeddings.

---

### 1. Feature Extraction with `HashingVectorizer`

```python
y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

np.random.seed(42)
vectorizer = HashingVectorizer(
    n_features=5000, stop_words="english", binary=True
)
X = vectorizer.fit_transform(documents)

```

* **`HashingVectorizer`**: Uses a deterministic hashing trick (MurmurHash3) to map words directly into a fixed-size feature vector of length 5,000 without needing to hold a complete dictionary in memory.
* **`binary=True`**: Creates binary indicators (one-hot presence vectors: $1$ if a word is present, $0$ otherwise) rather than frequency counts.
* **Target Array ($y$)**: Converts all ground truth training prices into a 1D NumPy float array.

---

### 2. Neural Network Architecture Definition

```python
class NeuralNetwork(nn.Module):

  def __init__(self, input_size):
    super(NeuralNetwork, self).__init__()
    self.layer1 = nn.Linear(input_size, 128)
    self.layer2 = nn.Linear(128, 64)
    self.layer3 = nn.Linear(64, 64)
    self.layer4 = nn.Linear(64, 64)
    self.layer5 = nn.Linear(64, 64)
    self.layer6 = nn.Linear(64, 64)
    self.layer7 = nn.Linear(64, 64)
    self.layer8 = nn.Linear(64, 1)
    self.relu = nn.ReLU()

  def forward(self, x):
    output1 = self.relu(self.layer1(x))
    output2 = self.relu(self.layer2(output1))
    output3 = self.relu(self.layer3(output2))
    output4 = self.relu(self.layer4(output3))
    output5 = self.relu(self.layer5(output4))
    output6 = self.relu(self.layer6(output5))
    output7 = self.relu(self.layer7(output6))
    output8 = self.layer8(output7)
    return output8

```

* **Input Layer (`layer1`)**: Projects the sparse 5,000-dimensional text vector into a 128-neuron hidden representation.
* **Hidden Layers (`layer2` to `layer7`)**: Sequential 64-neuron linear transformations with non-linear **ReLU** activations ($\text{ReLU}(z) = \max(0, z)$) to capture complex feature interactions.
* **Output Layer (`layer8`)**: A single linear unit with **no activation function**, directly outputting the continuous predicted price.

---

### 3. PyTorch Data Pipeline Setup

```python
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)  # Reshape to (N, 1)

# Validation split (1% holdout for loss monitoring)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_tensor, y_train_tensor, test_size=0.01, random_state=42
)

# Mini-batch DataLoader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Instantiate model & count parameters
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params:,}")

```

* **`y_train_tensor.unsqueeze(1)`**: Reshapes the target from `[N]` to `[N, 1]` to align matrix dimensions with the network's output layer.
* **`DataLoader(..., batch_size=64, shuffle=True)`**: Automatically batches the dataset into chunks of 64 and shuffles rows per epoch to improve gradient descent convergence.
* **`trainable_params`**: Calculates the total number of learnable weights and biases across all 8 layers.

---

### 4. Training Loop (The 4 Stages of Backpropagation)

```python
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 2

for epoch in range(EPOCHS):
  model.train()
  for batch_X, batch_y in tqdm(train_loader):
    optimizer.zero_grad()  # Reset stale gradients

    # --- The 4 Core Stages ---
    outputs = model(batch_X)  # 1. Forward pass
    loss = loss_function(outputs, batch_y)  # 2. Loss calculation (MSE)
    loss.backward()  # 3. Backward pass (Backprop)
    optimizer.step()  # 4. Weight update via Adam

  # Validation step
  model.eval()
  with torch.no_grad():
    val_outputs = model(X_val)
    val_loss = loss_function(val_outputs, y_val)

  print(
      f"Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss:"
      f" {val_loss.item():.3f}"
  )

```

1. **`optimizer.zero_grad()`**: Clears accumulated gradients from the previous iteration.
2. **Forward Pass**: Passes input mini-batch `batch_X` through the network layers.
3. **Loss Computation**: Evaluates Mean Squared Error $\frac{1}{B}\sum (\hat{y} - y)^2$.
4. **Backward Pass (`loss.backward()`)**: Calculates gradients of the loss with respect to every weight using the chain rule.
5. **Optimizer Step (`optimizer.step()`)**: Updates network weights using the **Adam** adaptive learning rate optimizer.
6. **Validation Evaluation**: Disables gradient tracking (`torch.no_grad()`) and checks loss against unseen validation samples to monitor convergence.

---

### 5. Single-Item Inference & Capstone Evaluation

```python
def neural_network(item: Item) -> float:
  model.eval()
  with torch.no_grad():
    # 1. Transform text summary to vector
    vector = vectorizer.transform([item.summary])
    # 2. Convert to FloatTensor
    tensor_in = torch.FloatTensor(vector.toarray())
    # 3. Predict price
    result = model(tensor_in)[0].item()
  # Clamp negative outputs to $0.00
  return max(0, result)


# Run evaluation on unseen test split
evaluate(neural_network, test)

```

* Sets model mode to `model.eval()` to freeze training-specific behaviors.
* Transforms raw `item.summary` using `vectorizer.transform()`, wraps the output in a PyTorch tensor, extracts the scalar via `.item()`, and applies `max(0, result)` non-negative bounding before reporting final test metrics.

In [12]:
y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

np.random.seed(42)
vectorizer = HashingVectorizer(
    n_features=5000, stop_words="english", binary=True
)
X = vectorizer.fit_transform(documents)

In [13]:
class NeuralNetwork(nn.Module):

  def __init__(self, input_size):
    super(NeuralNetwork, self).__init__()
    self.layer1 = nn.Linear(input_size, 128)
    self.layer2 = nn.Linear(128, 64)
    self.layer3 = nn.Linear(64, 64)
    self.layer4 = nn.Linear(64, 64)
    self.layer5 = nn.Linear(64, 64)
    self.layer6 = nn.Linear(64, 64)
    self.layer7 = nn.Linear(64, 64)
    self.layer8 = nn.Linear(64, 1)
    self.relu = nn.ReLU()

  def forward(self, x):
    output1 = self.relu(self.layer1(x))
    output2 = self.relu(self.layer2(output1))
    output3 = self.relu(self.layer3(output2))
    output4 = self.relu(self.layer4(output3))
    output5 = self.relu(self.layer5(output4))
    output6 = self.relu(self.layer6(output5))
    output7 = self.relu(self.layer7(output6))
    output8 = self.layer8(output7)
    return output8

In [14]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)  # Reshape to (N, 1)

# Validation split (1% holdout for loss monitoring)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_tensor, y_train_tensor, test_size=0.01, random_state=42
)

# Mini-batch DataLoader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Instantiate model & count parameters
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249

In [15]:
def neural_network(item: Item) -> float:
  model.eval()
  with torch.no_grad():
    # 1. Transform text summary to vector
    vector = vectorizer.transform([item.summary])
    # 2. Convert to FloatTensor
    tensor_in = torch.FloatTensor(vector.toarray())
    # 3. Predict price
    result = model(tensor_in)[0].item()
  # Clamp negative outputs to $0.00
  return max(0, result)


# Run evaluation on unseen test split
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$145 $26 $16 $26 $40 $188 $47 $35 $11 $275 $399 $332 $19 $36 $740 $15 $68 $45 $19 $40 $21 $200 $170 $30 $50 $19 $110 $29 $80 $6 $95 $15 $29 $16 $80 $292 $65 $36 $133 $15 $20 $132 $18 $12 $70 $13 $13 $16 $36 $49 $14 $30 $234 $46 $26 $30 $7 $12 $16 $44 $27 $55 $67 $20 $145 $35 $16 $229 $36 $13 $46 $12 $19 $27 $20 $13 $45 $8 $14 $27 $60 $23 $8 $43 $33 $14 $10 $890 $67 $28 $16 $18 $19 $30 $36 $21 $16 $16 $13 $9 $7 $15 $14 $70 $21 $13 $15 $110 $31 $184 $16 $27 $29 $19 $105 $34 $22 $26 $90 $39 $10 $108 $16 $12 $53 $17 $20 $36 $171 $40 $27 $50 $14 $76 $14 $35 $28 $40 $34 $42 $33 $36 $52 $12 $12 $17 $28 $19 $14 $10 $12 $156 $27 $160 $46 $29 $30 $70 $54 $20 $389 $13 $9 $13 $750 $21 $10 $14 $25 $13 $10 $79 $344 $39 $16 $370 $144 $44 $22 $17 $200 $37 $300 $12 $45 $22 $37 $60 $75 $40 $290 $17 $10 $43 $26 $46 $219 $35 $12 $40 

# Deep Learning Foundations: ReLU, Gradient Descent, and Backpropagation

In neural networks, training is fundamentally an optimization problem: we want to find the exact set of parameters (weights and biases) that minimizes the network's prediction error, measured by a **Loss function** $L$.

The mathematical engine that drives this learning process consists of three tightly coupled components:

1. **Derivatives (and Activation Functions like ReLU)**
2. **Gradient Descent**
3. **Backpropagation**

---

## 1. The Concept of the Derivative & The ReLU Case

### What is a Derivative?

In calculus, the **derivative** of a function measures its sensitivity to change:


$$\frac{df(z)}{dz} = \lim_{\Delta z \to 0} \frac{f(z + \Delta z) - f(z)}{\Delta z}$$


It tells us: *"If we make a tiny nudge to input $z$, how much and in what direction will output $f(z)$ change?"*

### The Rectified Linear Unit (ReLU)

Activation functions introduce non-linearity into neural networks, allowing them to learn complex patterns beyond simple linear regressions.

The **ReLU** activation function is defined piecewise as:


$$f(z) = \max(0, z) = \begin{cases} z & \text{if } z > 0 \\ 0 & \text{if } z \le 0 \end{cases}$$

```
   f(z) (ReLU Output)                  f'(z) (Derivative / Slope)
         |     /                             |
         |    /                            1 |       +--------- (Slope = 1)
         |   /                               |       |
  _______|__/________ z               _______|_______|_________ z
         0                                   0 | (Slope = 0)

```

### The Derivative of ReLU

Taking the derivative with respect to $z$:


$$f'(z) = \frac{df(z)}{dz} = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z < 0 \end{cases}$$

*(Note: At exactly $z = 0$, the function is technically non-differentiable. In deep learning frameworks like PyTorch or TensorFlow, a sub-gradient is used, defining $f'(0)$ as either $0$ or $1$.)*

### Practical Significance of ReLU's Derivative

* **Active Region ($z > 0$, Slope = 1):** The derivative is a constant $1$. It allows gradients to flow backwards through layers without shrinking. This largely solves the **Vanishing Gradient Problem** common to saturating activations like Sigmoid ($\sigma'(z) \le 0.25$) and Tanh ($\tanh'(z) \le 1.0$).
* **Inactive Region ($z < 0$, Slope = 0):** The derivative is $0$, completely blocking the gradient for that neuron. While this creates desirable **sparse activations** (fewer active neurons per inference), it can also cause the **"Dying ReLU"** problem if a neuron gets stuck outputting negative values for all dataset inputs.

---

## 2. What is Gradient Descent?

**Gradient Descent** is an iterative first-order optimization algorithm used to find the local minimum of the loss function $L(W)$.

### Intuition: The Foggy Mountain Analogy

Imagine standing on a mountain in dense fog. You cannot see the lowest valley (minimum error), but you can feel the slope of the ground beneath your feet. To reach the bottom:

1. You feel which way is downhill (the negative gradient).
2. You take a step in that direction.
3. You repeat the process until the ground flattens out.

### Mathematical Formulation

For a weight parameter $W$, the gradient $\frac{\partial L}{\partial W}$ represents the direction of steepest *ascent* (increase in loss). To decrease loss, we step in the opposite direction:

$$W_{\text{new}} = W_{\text{old}} - \alpha \cdot \frac{\partial L}{\partial W}$$

Where:

* $W_{\text{old}}$: Current parameter value
* $W_{\text{new}}$: Updated parameter value
* $\alpha$ (Alpha): **Learning Rate** (a hyperparameter controlling step size)
* $\frac{\partial L}{\partial W}$: **Partial Derivative** of the loss with respect to weight $W$

---

## 3. Backpropagation: Connecting Derivatives & Gradient Descent

Gradient descent tells us *how to update* weights once we know $\frac{\partial L}{\partial W}$. **Backpropagation** is the computational algorithm that actually *calculates* $\frac{\partial L}{\partial W}$ for every weight in a multi-layered network.

### The Chain Rule

In a neural network, weights in early layers are separated from the final loss through nested composite functions:


$$\text{Input } x \longrightarrow z = Wx + b \longrightarrow a = f(z) \longrightarrow \dots \longrightarrow \text{Loss } L$$

Using the calculus **Chain Rule**, the total sensitivity of the Loss $L$ to a specific weight $W$ is obtained by multiplying the local rates of change along the path:

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial a} \times \frac{\partial a}{\partial z} \times \frac{\partial z}{\partial W}$$

Breaking down each component:

1. $\frac{\partial L}{\partial a}$: **Upstream Gradient** (how loss changes with the activation output).
2. $\frac{\partial a}{\partial z}$: **Activation Derivative** ($f'(z) = 1$ or $0$ for ReLU).
3. $\frac{\partial z}{\partial W}$: **Local Input** (since $z = Wx + b$, $\frac{\partial z}{\partial W} = x$).

Thus, for a ReLU layer:


$$\frac{\partial L}{\partial W} = \begin{cases} \frac{\partial L}{\partial a} \cdot x & \text{if } z > 0 \\ 0 & \text{if } z \le 0 \end{cases}$$

---

## 4. Summary of the Complete Training Loop

```
========================================================================
[1] FORWARD PASS:
    Input x  ───► [ z = Wx + b ] ───► [ a = ReLU(z) ] ───► Loss L
                                                               │
                                                               ▼
[2] BACKWARD PASS (Backpropagation via Chain Rule):             │
    dL/dW <─── (x) <─── [ f'(z) = 1 or 0 ] <─── dL/da <────────┘
      │
      ▼
[3] PARAMETER UPDATE (Gradient Descent):
    W := W - α * (dL/dW)
========================================================================

```

1. **Forward Pass:** Compute linear combinations and activations forward to get the network output and compute the scalar loss $L$.
2. **Backward Pass (Backpropagation):** Traverse the computational graph in reverse, applying the chain rule to compute $\frac{\partial L}{\partial W}$ for every parameter using local derivatives (like ReLU's $f'(z)$).
3. **Optimization Step (Gradient Descent):** Apply the update rule $W \leftarrow W - \alpha \frac{\partial L}{\partial W}$ to adjust all weights towards lower overall error.

#### And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

In [16]:
def messages_for(item: Item) -> list[dict[str, str]]:
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [17]:
print(test[0].summary)

Title: Vestil DHHT‑500S Dual‑Handle Steel Hand Truck  
Category: Industrial Hand Trucks  
Brand: Vestil  
Description: A 500‑lb capacity steel hand truck featuring dual handles for superior control.  
Details: Equipped with 10‑inch pneumatic wheels, a 44‑½” height, 22‑½” width, 17‑½” depth, and a 13‑¾”×6‑½” nose 
plate.

In [18]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Vestil DHHT‑500S Dual‑Handle Steel Hand Truck  \nCategory: Industrial Hand Trucks  \nBrand: Vestil  \nDescription: A 500‑lb capacity steel hand truck featuring dual handles for superior control.  \nDetails: Equipped with 10‑inch pneumatic wheels, a 44‑½” height, 22‑½” width, 17‑½” depth, and a 13‑¾”×6‑½” nose plate.'}]

In [21]:
# The function for gpt-oss-20b (hosted on Groq)

def gpt_4__1_nano(item):
    model = f"groq/{os.environ['GROQ_MODEL']}"
    response = completion(model=model, api_key=os.environ["GROQ_API_KEY"], messages=messages_for(item))
    return response.choices[0].message.content

In [22]:
gpt_4__1_nano(test[0])

22:21:25 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-19 22:21:25,778 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:21:26 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-19 22:21:26,172 INFO LiteLLM Wrapper: Completed Call, calling success_handler


'$320'

In [23]:
test[0].price

144.96

In [24]:
evaluate(gpt_4__1_nano, test)

22:21:57 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-19 22:21:57,575 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:21:57 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-19 22:21:57,580 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:21:57 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-19 22:21:57,583 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:21:57 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-19 22:21:57,587 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:21:57 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


  0%|          | 0/200 [00:00<?, ?it/s]

2026-08-19 22:21:57,589 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:21:57 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-19 22:21:57,945 INFO LiteLLM Wrapper: Completed Call, calling success_handler
22:21:57 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-19 22:21:57,948 INFO LiteLLM Wrapper: Completed Call, calling success_handler
22:21:57 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-19 22:21:57,949 INFO LiteLLM Wrapper: Completed Call, calling success_handler
22:21:57 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-19 22:21:57,950 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:21:57 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-19 22:21:57,954 INFO LiteLLM 
LiteLLM completion

$135 $13 $12 $9 $45 

22:21:58 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-19 22:21:58,252 INFO LiteLLM Wrapper: Completed Call, calling success_handler
22:21:58 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-19 22:21:58,254 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:21:58 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-19 22:21:58,256 INFO LiteLLM Wrapper: Completed Call, calling success_handler
22:21:58 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-19 22:21:58,256 INFO LiteLLM Wrapper: Completed Call, calling success_handler
22:21:58 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-19 22:21:58,260 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:21:58 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM comple

...

$28 $1 $125 $18 $130 $18 $184 $10 $162 $12 $40 $5 $910 $3 $2 

22:22:08 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-19 22:22:08,296 INFO LiteLLM Wrapper: Completed Call, calling success_handler
22:22:08 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-19 22:22:08,299 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:22:08 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-19 22:22:08,313 INFO LiteLLM Wrapper: Completed Call, calling success_handler
22:22:08 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-19 22:22:08,315 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:22:08 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-19 22:22:08,380 INFO LiteLLM Wrapper: Completed Call, calling success_handler
22:22:08 - LiteLLM:INFO: utils.py:1470 - Wrapper: Comple

$156 $19 $17 $131 

22:22:08 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-19 22:22:08,957 INFO LiteLLM Wrapper: Completed Call, calling success_handler


$17 $23 $5 